# Observability analysis with the PlumeEnvironment_v3 `force`-physics dynamical system

This notebook follows the analysis pipeline of `observability_analysis_dynamic_wind.ipynb`
(load RL eval logs $\rightarrow$ decode control inputs $\rightarrow$ open-loop simulate & validate against
ground truth $\rightarrow$ sliding empirical observability matrix $\rightarrow$ Fisher information / minimum
error variance), but replaces the kinematic dynamical system with the **force-based system of
`fly_wind_example.ipynb`**, matched exactly to the physics model of `PlumeEnvironment_v3`
(`tamagotchi/env.py`) when `action_physics == 'force'`.

Reference for the dynamical system:

Floris van Breugel,
*A Nonlinear Observability Analysis of Ambient Wind Estimation with Uncalibrated Sensors, Inspired by Insect Neural Encoding*,
2021 60th IEEE CDC. DOI: 10.1109/CDC45484.2021.9683219

## Equivalence of the two models

PEv3 `force` physics integrates rigid-body dynamics in the **world frame**
(semi-implicit Euler, `physics_substeps` per env step, zero-order hold on thrust and wind):

$$
m\dot{\mathbf{v}} = R(\phi)\begin{bmatrix}T_{\parallel}\\T_{\perp}\end{bmatrix} - c\,(\mathbf{v}-\mathbf{w}),
\qquad
I\ddot{\phi} = \tau - k_{rot}\dot{\phi},
\qquad
\dot{\mathbf{p}} = \mathbf{v}
$$

Rewriting the translational equation in the body frame with
$v_{\parallel} = \mathbf{v}\cdot\hat{e}_{\parallel}(\phi)$, $v_{\perp} = \mathbf{v}\cdot\hat{e}_{\perp}(\phi)$
(the frame rotates with the body, adding the $\pm\dot\phi$ coupling terms) gives **exactly the
`fly_wind_example` system**:

$$
\dot{v}_{\parallel} = \tfrac{1}{m}(u_{\parallel} - C_{\parallel} a_{\parallel}) + v_{\perp}\dot{\phi},
\qquad
\dot{v}_{\perp} = \tfrac{1}{m}(u_{\perp} - C_{\perp} a_{\perp}) - v_{\parallel}\dot{\phi},
\qquad
\ddot{\phi} = \tfrac{1}{I}(u_{\phi} - C_{\phi}\dot{\phi})
$$

with air velocity
$a_{\parallel} = v_{\parallel} - w\cos(\phi-\zeta)$, $a_{\perp} = v_{\perp} + w\sin(\phi-\zeta)$.

### Parameter mapping (fly_wind_example $\leftrightarrow$ PEv3 `conf/physics/fly.yaml`)

| fly_wind_example | PEv3 force physics | value (SI, fly.yaml) |
|---|---|---|
| $m$ | `mass` | 1.0e-6 kg |
| $C_{\parallel} = C_{\perp}$ | `drag` (isotropic) | 1.93e-6 N s/m |
| $I$ | `inertia` | 5.2e-13 kg m² |
| $C_{\phi}$ | `k_rot` | 1.61e-12 N m s |
| $k_{m_1}=k_{m_3}=k_{m_4}$ | (none) | 1 |
| $k_{m_2}$ | (none) | **0** |
| $u_{\parallel}$ | `action[0] * T_para_max` | $[0,\,2.9\times10^{-6}]$ N |
| $u_{\perp}$ | `(action[1]-0.5) * 2 * T_perp_max` | $\pm 1.3\times10^{-6}$ N |
| $u_{\phi}$ | `(action[2]-0.5) * 2 * tau_max` | $\pm 1.41\times10^{-11}$ N m |

### Required modifications to fly_wind_example for exact compatibility

1. **$k_{m_2}=0$** — PEv3 has no thrust$\to$torque cross-coupling term.
2. **Isotropic drag** — PEv3 uses a single `drag` for both body axes, so $C_{\parallel}=C_{\perp}$.
3. **No altitude** — PEv3 is planar; drop state $z$ and the optic-flow measurement $r=g/z$
   (its "ventral optic flow" cue is the egocentric course direction $\psi$, not a flow magnitude).
4. **Use `conf/physics/fly.yaml` coefficients in SI units** — not the mg/mm-scaled literature
   parameters used in fly_wind_example.
5. **Measurements match the agent's observations** (with `apparent_wind=True`):
   heading $\phi$, egocentric course direction $\psi=\arctan(v_{\perp}/v_{\parallel})$,
   and the egocentric apparent-wind *vector* $-(a_{\parallel}, a_{\perp})$
   — equivalently apparent airflow angle $\gamma=\arctan(a_{\perp}/a_{\parallel})$ and magnitude $a$
   as in fly_wind_example. Note this is a structural change from the kinematic
   `observability_analysis_dynamic_wind` system, where `appWind` $=-u_{\parallel}$ was a pure copy
   of the input and carried no state information; under force physics the apparent-wind
   measurement genuinely depends on $(v_{\parallel}, v_{\perp}, \phi, w, \zeta)$.
6. **Wind dynamics as inputs** — as in the dynamic-wind notebook, $\dot\zeta = u_{\dot\zeta}$ and
   $\dot{w} = u_{\dot w}$ track logged wind changes. Unlike the kinematic system, **no**
   $\dot w/\dot\zeta$ coupling terms appear in $\dot v_{\parallel}/\dot v_{\perp}$: ground velocity
   is a true integrator state here, and wind enters only through the drag force.
7. **$\dot\phi$ is a state, not an input** — PEv3 `force` integrates `ang_vel`; eval logs of
   force-physics agents store it (info `ang_vel` / traj_df `angular_velocity`), which provides
   the initial condition.

### Known (small) discretization difference

PEv3 integrates with semi-implicit Euler at `sub_dt = dt/physics_substeps` (0.005 s for
`dt=0.04, physics_substeps=8`); pybounds/do-mpc integrates the continuous ODE (IDAS) with
zero-order-hold inputs per `dt`. The env converges to this ODE at first order in `sub_dt`;
with the fly time constants ($\tau_{lin}=m/c\approx0.52$ s, $\tau_{rot}=I/k_{rot}\approx0.32$ s $\gg$ 0.005 s)
the mismatch is ~0.3–0.5% of signal scale over a 300-step episode (verified numerically below).

Also note for exact replay: episodes must be logged with `action_delay_const=None`,
`action_latency=None`, `obs_noise=0`, and no `flip_ventral_optic_flow`/`double_drift`
(those alter the applied action or the sensed course direction relative to the logs).


In [ ]:
import time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pybounds import Simulator, SlidingEmpiricalObservabilityMatrix, FisherObservability, SlidingFisherObservability, ObservabilityMatrixImage, colorline

def log_shapes(pytree):
    # print shapes of arrays in a (possibly nested) dict
    for key, leaf in pytree.items():
        if isinstance(leaf, np.ndarray):
            print(f"Path: {key}, Shape: {leaf.shape}")

## Physics coefficients

Loaded from `tamagotchi/conf/physics/fly.yaml` when available (so the notebook stays in sync
with the coefficients the env actually trained with), with a hardcoded fallback.
**These must match the `physics=<...>` config the eval logs were generated with** —
swap in `drone.yaml`/`drone_low_acc.yaml` values for drone agents.

In [ ]:
# conf/physics/fly.yaml (SI units) - fallback values
force_physics = {
    'T_para_max': 2.90e-6,  # [N] parallel thrust max; action[0] in [0,1] -> [0, T_para_max]
    'T_perp_max': 1.30e-6,  # [N] perpendicular thrust max; action[1] -> [-T_perp_max, T_perp_max]
    'tau_max':    1.41e-11, # [N*m] yaw torque max; action[2] -> [-tau_max, tau_max]
    'mass':       1.0e-6,   # [kg]
    'drag':       1.93e-6,  # [N*s/m] isotropic linear drag on airspeed
    'inertia':    5.2e-13,  # [kg*m^2] yaw moment of inertia
    'k_rot':      1.61e-12, # [N*m*s] yaw damping
    'physics_substeps': 8,  # env substeps per env step (dt=0.04 -> sub_dt=0.005)
}

# Prefer the yaml shipped with tamagotchi if importable
try:
    import yaml
    from pathlib import Path
    import tamagotchi
    yaml_path = Path(tamagotchi.__file__).parent / 'conf' / 'physics' / 'fly.yaml'
    with open(yaml_path) as f_yaml:
        loaded = yaml.safe_load(f_yaml)
    force_physics.update({k: loaded[k] for k in force_physics if k in loaded})
    print(f"Loaded force physics coefficients from {yaml_path}")
except Exception as e:
    print(f"Using hardcoded fly.yaml coefficients ({e})")

M      = force_physics['mass']
C_PARA = force_physics['drag']   # isotropic in PEv3: C_para = C_perp = drag
C_PERP = force_physics['drag']
I_YAW  = force_physics['inertia']
C_PHI  = force_physics['k_rot']
print(f"tau_lin = {M/C_PARA:.4g} s, tau_rot = {I_YAW/C_PHI:.4g} s")
force_physics

## Define dynamics function
The dynamics function takes in a list of states $X$ and a list of inputs $U$ and outputs the derivative of the states.

States: $x, y, v_{\parallel}, v_{\perp}, \phi, \dot{\phi}, w, \zeta$ — the fly_wind_example primary
states minus altitude $z$ (planar env) and minus the inertia/damping/motor-calibration parameter
states (fixed constants of the env physics).

Inputs: body-frame thrust $u_{\parallel}, u_{\perp}$ and yaw torque $u_{\phi}$ (decoded from logged
actions), plus $u_{\dot\zeta}, u_{\dot w}$ to track logged wind changes (dynamic-wind trick).

In [ ]:
state_names = [
                'x',        # x position [m]
                'y',        # y position [m]
                'v_para',   # parallel ground velocity [m/s]
                'v_perp',   # perpendicular ground velocity [m/s]
                'phi',      # heading [rad]
                'phi_dot',  # angular velocity [rad/s] - integrator state under force physics
                'w',        # ambient wind speed [m/s]
                'zeta',     # ambient wind angle [rad]
                ]

input_names = [
                'u_para',     # parallel thrust force [N]
                'u_perp',     # perpendicular thrust force [N]
                'u_phi',      # yaw torque [N*m]
                'u_zeta_dot', # wind direction rate [rad/s] - for discontinuous wind direction change
                'u_w_dot',    # wind speed rate [m/s^2]
                ]

def f(X, U):
    '''
    Return Xdot given X and U.

    Body-frame form of PlumeEnvironment_v3 action_physics == 'force':
        m*v_dot   = R(phi)[u_para, u_perp] - drag*(v - wind)   (world frame)
        I*phi_ddot = u_phi - k_rot*phi_dot
    with km1 = km3 = km4 = 1, km2 = 0, C_para = C_perp = drag relative to fly_wind_example.
    '''
    # States
    x, y, v_para, v_perp, phi, phi_dot, w, zeta = X

    # Inputs
    u_para, u_perp, u_phi, u_zeta_dot, u_w_dot = U

    # Air velocity (body frame)
    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)

    # Acceleration (thrust + linear drag on airspeed + rotating-frame coupling)
    v_para_dot = ((u_para - C_PARA * a_para) / M) + (v_perp * phi_dot)
    v_perp_dot = ((u_perp - C_PERP * a_perp) / M) - (v_para * phi_dot)

    # Angular acceleration
    phi_ddot = (u_phi - C_PHI * phi_dot) / I_YAW

    # Wind dynamics driven by inputs (constant wind: both zero)
    w_dot = u_w_dot
    zeta_dot = u_zeta_dot

    # Position dynamics
    x_dot = v_para * np.cos(phi) - v_perp * np.sin(phi)
    y_dot = v_para * np.sin(phi) + v_perp * np.cos(phi)

    # Package and return xdot
    X_dot = [x_dot, y_dot, v_para_dot, v_perp_dot, phi_dot, phi_ddot, w_dot, zeta_dot]

    return X_dot

## Define measurement function
The measurement function takes in a list of states $X$ and a list of inputs $U$ and outputs the measurements $Y$.

These correspond to the PEv3 observation channels (`apparent_wind=True`):
* `phi` — heading (obs `head_x/y`)
* `psi` — egocentric course direction / drift angle (obs `course_x/y`, the "ventral optic flow" cue)
* apparent wind (obs `wind_x/y` = $-$air velocity in the body frame), provided in two equivalent
  parameterizations so either can be selected downstream via `o_sensors`:
  * `gamma`, `a` — apparent airflow angle & magnitude (fly_wind_example convention;
    env obs vector = $a\,[\cos(\gamma+\pi), \sin(\gamma+\pi)]$)
  * `appW_para`, `appW_perp` $= -(a_{\parallel}, a_{\perp})$ — the vector components the agent
    actually receives (well-conditioned near zero airspeed, matches per-channel additive obs noise)

Odor is part of the RL observation but not a function of these states, so it is not modeled.
No $g$ or $r=g/z$: the agent senses course *direction* only, and the env has no altitude.

In [ ]:
measurement_names = ['phi', 'psi', 'gamma', 'a', 'appW_para', 'appW_perp']
def h(X, U):
    '''
    Measurement function - input is the state and control input; output is the measurement.
    '''
    # States
    x, y, v_para, v_perp, phi, phi_dot, w, zeta = X

    # Inputs
    u_para, u_perp, u_phi, u_zeta_dot, u_w_dot = U

    # Air velocity (body frame)
    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)
    a = np.sqrt(a_para ** 2 + a_perp ** 2)
    gamma = np.arctan2(a_perp, a_para)  # air velocity angle

    # Apparent wind vector sensed by the agent (egocentric) = -air velocity
    appW_para = -a_para
    appW_perp = -a_perp

    # Course direction in fly reference frame (drift angle)
    psi = np.arctan2(v_perp, v_para)

    # Unwrap angles s.t. they are continuous over the time-series
    if np.array(phi).ndim > 0:
        if np.array(phi).shape[0] > 1:
            phi = np.unwrap(phi)
            psi = np.unwrap(psi)
            gamma = np.unwrap(gamma)

    # Measurements
    Y = [phi, psi, gamma, a, appW_para, appW_perp]

    # Return measurement
    return Y

## Set time-step
Must match the env step (`config.env` dt used at training/eval).

In [ ]:
dt = 0.04  # [s]

# Verify equivalence with PlumeEnvironment_v3 `force` physics (synthetic, no eval data needed)

Replicates the env's world-frame semi-implicit Euler integrator
(`env.py`, `step()`, `action_physics == 'force'` branch) on a random squashed action sequence
with a mid-episode wind switch, and compares against a high-accuracy integration of the
body-frame ODE `f` defined above. Also shows first-order convergence of the env integrator
to the ODE as `physics_substeps` increases.

In [ ]:
from scipy.integrate import solve_ivp

def pev3_force_replica(actions, wind_xy, pc, dt, x0=None):
    '''
    Line-for-line replica of the PEv3 force-physics integrator (world frame).
    actions: [T, 3] squashed to [0, 1]; wind_xy: [T, 2] ambient wind per env step (ZOH).
    x0: optional dict with keys pos, vel, ang, ang_vel (defaults = env reset: vel=0, ang_vel=0).
    Returns [T, 6]: x, y, vel_x, vel_y, phi, phi_dot AFTER each step.
    '''
    x0 = x0 or {}
    pos = np.array(x0.get('pos', [0.0, 0.0]), dtype=np.float64)
    vel = np.array(x0.get('vel', [0.0, 0.0]), dtype=np.float64)
    ang = float(x0.get('ang', 0.0))
    ang_vel = float(x0.get('ang_vel', 0.0))
    out = []
    sub_dt = dt / pc['physics_substeps']
    for a, wind in zip(actions, wind_xy):
        T_par = float(a[0]) * pc['T_para_max']
        T_per = (float(a[1]) - 0.5) * 2.0 * pc['T_perp_max']
        tau   = (float(a[2]) - 0.5) * 2.0 * pc['tau_max']
        for _ in range(pc['physics_substeps']):
            c, s = np.cos(ang), np.sin(ang)
            T_world = np.array([c * T_par - s * T_per,
                                s * T_par + c * T_per])
            v_air = vel - wind
            vel = vel + (T_world - pc['drag'] * v_air) / pc['mass'] * sub_dt  # semi-implicit Euler
            pos = pos + vel * sub_dt   # uses the *updated* velocity
            ang_vel = ang_vel + (tau - pc['k_rot'] * ang_vel) / pc['inertia'] * sub_dt
            ang = ang + ang_vel * sub_dt
        out.append([pos[0], pos[1], vel[0], vel[1], ang, ang_vel])
    return np.array(out)

def body_ode_sim(actions, wind_xy, pc, dt, x0=None):
    '''High-accuracy integration of f (body frame) with the same per-step ZOH inputs.'''
    x0 = x0 or {}
    X = np.array([*x0.get('pos', [0.0, 0.0]), 0.0, 0.0, x0.get('ang', 0.0),
                  x0.get('ang_vel', 0.0), 0.0, 0.0], dtype=np.float64)
    out = []
    for a, wind in zip(actions, wind_xy):
        u_para = float(a[0]) * pc['T_para_max']
        u_perp = (float(a[1]) - 0.5) * 2.0 * pc['T_perp_max']
        u_phi  = (float(a[2]) - 0.5) * 2.0 * pc['tau_max']
        X[6] = np.hypot(wind[0], wind[1])       # w (ZOH within step, like the env)
        X[7] = np.arctan2(wind[1], wind[0])     # zeta
        U = [u_para, u_perp, u_phi, 0.0, 0.0]
        sol = solve_ivp(lambda t, Xt: f(Xt, U), (0, dt), X, rtol=1e-10, atol=1e-12)
        X = sol.y[:, -1]
        out.append(X.copy())
    return np.array(out)

# Random squashed actions + a wind switch halfway through
rng = np.random.default_rng(0)
n_check = 300
actions_check = (np.tanh(rng.normal(0, 1.0, size=(n_check, 3))) + 1) / 2
zeta_check = np.where(np.arange(n_check) < n_check // 2, np.pi, np.pi - np.pi / 4)
w_check = 0.5 + 0.05 * np.sin(np.arange(n_check) * 0.1)
wind_check = np.stack([w_check * np.cos(zeta_check), w_check * np.sin(zeta_check)], axis=1)

A = pev3_force_replica(actions_check, wind_check, force_physics, dt, x0={'ang': 0.7})
B = body_ode_sim(actions_check, wind_check, force_physics, dt, x0={'ang': 0.7})

# body -> world for comparison
vx_B = B[:, 2] * np.cos(B[:, 4]) - B[:, 3] * np.sin(B[:, 4])
vy_B = B[:, 2] * np.sin(B[:, 4]) + B[:, 3] * np.cos(B[:, 4])
for name, env_sig, ode_sig in [('x', A[:, 0], B[:, 0]), ('y', A[:, 1], B[:, 1]),
                               ('vel_x', A[:, 2], vx_B), ('vel_y', A[:, 3], vy_B),
                               ('phi', A[:, 4], B[:, 4]), ('phi_dot', A[:, 5], B[:, 5])]:
    err = np.max(np.abs(env_sig - ode_sig))
    scale = max(np.max(np.abs(env_sig)), 1e-12)
    print(f"{name:8s} max|env - ODE| = {err:.3e}  (signal scale {scale:.3f}, rel {err/scale:.2e})")

print("\nConvergence of the env integrator toward the ODE (first order in sub_dt):")
for n_sub in [1, 2, 4, 8, 32, 128]:
    Ai = pev3_force_replica(actions_check, wind_check, dict(force_physics, physics_substeps=n_sub),
                            dt, x0={'ang': 0.7})
    print(f"  substeps={n_sub:4d}  |final position error| = {np.linalg.norm(Ai[-1, :2] - B[-1, :2]):.3e} m")

# Measurement mapping: agent's apparent-wind obs (-air velocity, rotated into body frame)
# must equal (appW_para, appW_perp) = -(a_para, a_perp) from h. Evaluate both on the SAME
# (env) trajectory so this isolates the mapping - it should be exact to machine precision.
c, s = np.cos(A[:, 4]), np.sin(A[:, 4])
air_x, air_y = A[:, 2] - wind_check[:, 0], A[:, 3] - wind_check[:, 1]
appW_env = np.stack([-(c * air_x + s * air_y), -(-s * air_x + c * air_y)], axis=1)
v_para_A = c * A[:, 2] + s * A[:, 3]   # env world-frame velocity -> body frame
v_perp_A = -s * A[:, 2] + c * A[:, 3]
w_A = np.hypot(wind_check[:, 0], wind_check[:, 1])
zeta_A = np.arctan2(wind_check[:, 1], wind_check[:, 0])
a_para_A = v_para_A - w_A * np.cos(A[:, 4] - zeta_A)
a_perp_A = v_perp_A + w_A * np.sin(A[:, 4] - zeta_A)
print(f"\napparent wind obs vs h (same trajectory, should be ~machine precision): max err = "
      f"{np.max(np.abs(appW_env - np.stack([-a_para_A, -a_perp_A], axis=1))):.3e}")

# Create simulator object

In [ ]:
simulator = Simulator(f, h, dt=dt, state_names=state_names, input_names=input_names, measurement_names=measurement_names)

# Can also set the number of state (n) & inputs (m) instead of state & input names
# simulator = Simulator(f, h, dt=dt, n=len(state_names), m=len(input_names))

# Load inference-time trajectories to pull out control signals

The eval logs must come from an agent trained/evaluated with `action_physics='force'`
(3-dim actions). Point `log_fname` at such a pkl.

In [ ]:
log_fname = '/src/data/wind_sensing/CHANGE_ME_force_physics_agent/eval/plume_XXXX/noisy3x5b5.pkl'
# load pkl file
import pickle
with open(log_fname, 'rb') as f_handle:
    episode_logs = pickle.load(f_handle)
print('Loaded episode logs from', log_fname)
print('Number of episodes:', len(episode_logs))
print('Episodes contain:', episode_logs[0].keys())

In [ ]:
eps_idx = 1  # which episode to analyze
raw_actions = episode_logs[eps_idx]['actions']
raw_actions = np.stack(raw_actions)
assert raw_actions.shape[1] == 3, (
    f"Expected 3-dim force-physics actions [T_par, T_perp, tau], got {raw_actions.shape[1]}-dim. "
    "These logs are not from an action_physics='force' agent.")

def squash_and_scale_actions_force(raw_actions, pc):
    '''
    Decode logged raw policy outputs into the physical control inputs of the force model.
    Mirrors env.py step():
      1. squash: (tanh(x) + 1) / 2 -> [0, 1]
      2. scale:  u_para = a0 * T_para_max              in [0, T_para_max]
                 u_perp = (a1 - 0.5) * 2 * T_perp_max  in [-T_perp_max, T_perp_max]
                 u_phi  = (a2 - 0.5) * 2 * tau_max     in [-tau_max, tau_max]

    Timing (same convention as observability_analysis_dynamic_wind): infos[i] logs the state
    AFTER action[i] was applied, so the transition s_i -> s_{i+1} uses action[i+1]. We simulate
    from the first logged state, so the first action is omitted.
    '''
    if type(raw_actions) is list:
        raw_actions = np.stack(raw_actions)
    actions = np.clip((np.tanh(raw_actions) + 1) / 2, 0.0, 1.0)  # squash, per env step()

    u_sim = {'u_para': actions[1:, 0] * pc['T_para_max'],
             'u_perp': (actions[1:, 1] - 0.5) * 2.0 * pc['T_perp_max'],
             'u_phi':  (actions[1:, 2] - 0.5) * 2.0 * pc['tau_max']}
    squashed = actions  # keep full-length squashed actions for the integrator replay check
    print('u_sim shapes', u_sim['u_para'].shape, u_sim['u_perp'].shape, u_sim['u_phi'].shape)
    return u_sim, squashed

u_sim, squashed_actions = squash_and_scale_actions_force(raw_actions, force_physics)

# Pull out ground truth states and measurements to be simulated

In [ ]:
import tamagotchi.eval.log_analysis as log_analysis
number_of_eps = 240  # pull all episodes
dataset = 'noisy3x5b5'
# dataset = 'constantx5b5'
exp_folder = 'eval'
model_fname = log_fname.split('/eval/')[0] + '/weights/' + log_fname.split('/')[-2] + '.pt'

# based on open_loop_perturbation.py
eval_folder = model_fname.replace('weights', exp_folder).replace('.pt', '/')
selected_df = log_analysis.get_selected_df(eval_folder, [dataset],
                                        n_episodes_home=number_of_eps,
                                        n_episodes_other=number_of_eps,
                                        balanced=False,
                                        oob_only=False,
                                        verbose=True)

traj_df_stacked, stacked_neural_activity = log_analysis.get_traj_and_activity_and_stack_them(selected_df,
                                                                                            obtain_neural_activity = True,
                                                                                            obtain_traj_df = True,
                                                                                            get_traj_tmp = True,
                                                                                            extended_metadata = True)

epoch_traj_df = traj_df_stacked[traj_df_stacked['ep_idx'] == eps_idx]
epoch_latent_activity = stacked_neural_activity[epoch_traj_df.index]

In [ ]:
gt_dict = {'x': [], 'y': [], 'v_para': [], 'v_perp': [], 'phi': [], 'phi_dot': [], 'w': [], 'zeta': [],
           'psi_ego_course_dir': [], 'v_allo': []}

gt_dict['x'] = epoch_traj_df['loc_x'].values
gt_dict['y'] = epoch_traj_df['loc_y'].values
gt_dict['phi'] = np.angle(epoch_traj_df['agent_angle_x'] + 1j*epoch_traj_df['agent_angle_y'], deg=False)
gt_dict['phi_dot'] = epoch_traj_df['angular_velocity'].values  # logged env state under force physics (info 'ang_vel')
gt_dict['w'] = np.round(epoch_traj_df['wind_speed_ground'].values, 3)
gt_dict['zeta'] = epoch_traj_df['wind_angle_ground_theta'].values  # normalized by pi and then shifted to 0-1
gt_dict['psi_ego_course_dir'] = epoch_traj_df['ego_course_direction_theta'].values  # normalized by pi and then shifted to 0-1
gt_dict['v_allo'] = np.stack(epoch_traj_df['allo_ground_velocity'].values)  # true integrator state under force physics
# scale angles from 0-1 to -pi to pi
gt_dict['zeta'] = np.pi * (2*gt_dict['zeta'] - 1)
gt_dict['psi_ego_course_dir'] = np.pi * (2*gt_dict['psi_ego_course_dir'] - 1)

# body-frame ground velocity components (rotate the logged world-frame ground velocity by -phi)
g_mag = np.linalg.norm(gt_dict['v_allo'], axis=1)
gt_dict['v_para'] = np.cos(gt_dict['psi_ego_course_dir']) * g_mag  # cos(psi) * g = v_para
gt_dict['v_perp'] = np.sin(gt_dict['psi_ego_course_dir']) * g_mag  # sin(psi) * g = v_perp

# wind-change inputs (aligned with u_sim: transition s_i -> s_{i+1} uses the wind logged at i+1)
u_sim['u_zeta_dot'] = np.diff(np.unwrap(gt_dict['zeta'])) / dt
u_sim['u_w_dot'] = np.diff(gt_dict['w']) / dt
log_shapes(gt_dict)  # expected to have one more time point at the end
log_shapes(u_sim)    # 0th control signal is omitted

# Compatibility check: replay logged actions through the PEv3 force integrator

Before involving pybounds at all, feed the decoded actions + logged wind through the
line-for-line integrator replica. This should reproduce the logged trajectory to numerical
precision — it validates the action decoding, the coefficient set, and the timing alignment.
If this fails, fix it here (wrong physics yaml, action delay/latency enabled at eval, or an
alignment off-by-one) before interpreting any observability result.

In [ ]:
wind_xy_gt = np.stack([gt_dict['w'] * np.cos(gt_dict['zeta']),
                       gt_dict['w'] * np.sin(gt_dict['zeta'])], axis=1)

replay = pev3_force_replica(
    squashed_actions[1:], wind_xy_gt[1:], force_physics, dt,
    x0={'pos': [gt_dict['x'][0], gt_dict['y'][0]],
        'vel': gt_dict['v_allo'][0],
        'ang': gt_dict['phi'][0],
        'ang_vel': gt_dict['phi_dot'][0]})

for name, sim_sig, gt_sig in [('x', replay[:, 0], gt_dict['x'][1:]),
                              ('y', replay[:, 1], gt_dict['y'][1:]),
                              ('vel_x', replay[:, 2], gt_dict['v_allo'][1:, 0]),
                              ('vel_y', replay[:, 3], gt_dict['v_allo'][1:, 1]),
                              ('phi', replay[:, 4], gt_dict['phi'][1:]),
                              ('phi_dot', replay[:, 5], gt_dict['phi_dot'][1:])]:
    diff = np.asarray(sim_sig) - np.asarray(gt_sig)
    if name == 'phi':  # compare as angles to avoid spurious 2*pi wrap differences
        diff = np.arctan2(np.sin(diff), np.cos(diff))
    err = np.max(np.abs(diff))
    print(f"{name:8s} max|replay - logged| = {err:.3e}  ({'OK' if err < 1e-6 else 'CHECK ALIGNMENT / COEFFS'})")

# Simulate with control inputs to see if it matches ground truth

Now the pybounds simulator (continuous-time IDAS integration). Expect a close but not exact
match to ground truth — the residual is the env's semi-implicit-Euler discretization error
quantified in the synthetic verification above.

In [ ]:
x0 = {'x': gt_dict['x'][0], 'y': gt_dict['y'][0],
      'v_para': gt_dict['v_para'][0], 'v_perp': gt_dict['v_perp'][0],
      'phi': gt_dict['phi'][0], 'phi_dot': gt_dict['phi_dot'][0],
      'w': gt_dict['w'][0], 'zeta': gt_dict['zeta'][0]}

In [ ]:
t_sim, x_sim, u_sim, y_sim = simulator.simulate(x0=x0, mpc=False, u=u_sim, return_full_output=True)

# Plot the ground truth and the simulated trajectories

In [ ]:
gt_dict['phi'] = np.unwrap(gt_dict['phi'])
fig, axs = plt.subplots(len(state_names), 1, figsize=(10, 2*len(state_names)))
for i, state_name in enumerate(state_names):
    axs[i].plot(t_sim, x_sim[state_name], label='sim')
    axs[i].plot(t_sim, gt_dict[state_name][:-1], label='gt')
    axs[i].set_title(state_name)
    axs[i].legend()
plt.tight_layout()
plt.show()

# Observability

## Construct observability matrix in sliding windows

In [ ]:
w = 10  # window size, set to None to use entire time-series as one window

In [ ]:
# Construct O in sliding windows
st = time.time()
SEOM = SlidingEmpiricalObservabilityMatrix(simulator, t_sim, x_sim, u_sim, w=w, eps=1e-6)
et = time.time()
print('elapsed time:', et-st)

In [ ]:
# Get O's
O_sliding = SEOM.get_observability_matrix()

In [ ]:
n_window = len(O_sliding)
print(n_window, 'windows')

In [ ]:
# Visualize first sliding observability matrix, this will throw errors if O is too big
OI = ObservabilityMatrixImage(O_sliding[0], cmap='bwr')
OI.plot(scale=1.0)

## Compute Fisher information matrix & inverse for first window

In [ ]:
# Set each sensor noise level
sensor_noise = {'phi': 0.0001, 'psi': 0.0001, 'gamma': 0.0001, 'a': 0.0001,
                'appW_para': 0.0001, 'appW_perp': 0.0001}

In [ ]:
SEOM.O_df_sliding[0]

In [ ]:
# Compute the Fisher information & Chernoff inverse
FO = FisherObservability(SEOM.O_df_sliding[0], R=None, sensor_noise_dict=sensor_noise, lam=1e-12)

# Can also set R directly as matrix or as scalar
# FO = FisherObservability(SEOM.O_df_sliding[0], R=0.1*np.eye(O_sliding[0].shape[0]), lam=1e-6)
# FO = FisherObservability(SEOM.O_df_sliding[0], R=0.1, lam=1e-6)

In [ ]:
# Get the Fisher information, inverse, and R matrix
F, F_inv, R = FO.get_fisher_information()
F_inv

## Compute Fisher information matrix & inverse for each sliding window

`o_sensors` selects which measurements the agent is assumed to have. The set below
(`phi`, `psi`, `gamma`, `a`) is the angle/magnitude parameterization of exactly what the
PEv3 agent observes (heading, course direction, apparent wind). Swap `['gamma', 'a']` for
`['appW_para', 'appW_perp']` to use the vector parameterization instead — do not include
both at once, they are the same physical sensor.

In [ ]:
# Choose sensors to use from O
o_sensors = ['phi', 'psi', 'gamma', 'a']
# o_sensors = ['phi', 'psi', 'appW_para', 'appW_perp']  # vector parameterization of apparent wind

# Choose states to use from O
o_states = [
                # 'x',       # x position [m]
                # 'y',       # y position [m]
                'v_para',    # parallel ground velocity [m/s]
                'v_perp',    # perpendicular ground velocity [m/s]
                # 'phi',     # heading [rad] - directly measured
                'phi_dot',   # angular velocity [rad/s]
                'w',         # ambient wind speed [m/s]
                'zeta',      # ambient wind angle [rad]
                ]

# Choose time-steps to use from O
window_size = 10
o_time_steps = np.arange(0, window_size, step=1)

In [ ]:
# Compute the Fisher information & inverse for each window and store the minimum error variance
SFO = SlidingFisherObservability(SEOM.O_df_sliding, time=SEOM.t_sim, lam=1e-6, R=None, sensor_noise_dict=sensor_noise,
                                 states=o_states, sensors=o_sensors, time_steps=o_time_steps, w=None)

In [ ]:
# Pull out minimum error variance, 'time' column is the time vector shifted forward by w/2 and 'time_initial' is the original time
EV_aligned = SFO.get_minimum_error_variance()

In [ ]:
EV_aligned

In [ ]:
# Visualize observability matrix subset
OI = ObservabilityMatrixImage(SFO.FO[0].O)
OI.plot()

# Plot error variance as color on state time-series

In [ ]:
EV_no_nan = EV_aligned.fillna(method='bfill').fillna(method='ffill')

In [ ]:
states = [est_state_var for est_state_var in list(EV_no_nan.keys()) if est_state_var != 'time' and est_state_var != 'time_initial']
n_state = len(states)

fig, ax = plt.subplots(n_state, 2, figsize=(6, n_state*2), dpi=150)
ax = np.atleast_2d(ax)

cmap = 'inferno_r'

min_ev = np.min(EV_no_nan.iloc[:, 2:].values)
max_ev = np.max(EV_no_nan.iloc[:, 2:].values)

log_tick_high = int(np.ceil(np.log10(max_ev)))
log_tick_low = int(np.floor(np.log10(min_ev)))
cnorm = mpl.colors.LogNorm(10**log_tick_low, 10**log_tick_high)

for n, state_name in enumerate(states):
    colorline(x_sim['x'], x_sim['y'], EV_no_nan[state_name].values, ax=ax[n, 0], cmap=cmap, norm=cnorm)
    colorline(t_sim, EV_no_nan[state_name].values, EV_no_nan[state_name].values, ax=ax[n, 1], cmap=cmap, norm=cnorm)

    # Colorbar
    cax = ax[n, -1].inset_axes([1.03, 0.0, 0.04, 1.0])
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax,
                        ticks=np.logspace(log_tick_low, log_tick_high, log_tick_high-log_tick_low + 1))
    cbar.set_label('min. EV: ' + state_name, rotation=270, fontsize=7, labelpad=8)
    cbar.ax.tick_params(labelsize=6)

    ax[n, 0].set_ylim(np.min(x_sim['y']) - 0.01, np.max(x_sim['y']) + 0.01)
    ax[n, 0].set_xlim(np.min(x_sim['x']) - 0.01, np.max(x_sim['x']) + 0.01)
    ax[n, 0].set_ylabel('y', fontsize=7)
    ax[n, 0].set_xlabel('x', fontsize=7)
    ax[n, 0].set_aspect(1.0)

    ax[n, 1].set_ylim(10**log_tick_low, 10**log_tick_high)
    ax[n, 1].set_yscale('log')
    ax[n, 1].set_ylabel('min. EV: ' + state_name, fontsize=7)
    ax[n, 1].set_yticks(np.logspace(log_tick_low, log_tick_high, log_tick_high-log_tick_low + 1))


for a in ax.flat:
    a.tick_params(axis='both', labelsize=6)

for a in ax[:, 1]:
    a.set_xlabel('time (s)', fontsize=7)
    a.set_xlim(-0.1, t_sim[-1] + 0.1)

fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.3, hspace=0.4)

plt.show()

# Downstream analyses

`EV_aligned` / `EV_no_nan` have the same format as in `observability_analysis_dynamic_wind.ipynb`
(columns `time`, `time_initial`, one min-EV column per state in `o_states`), so the downstream
sections of that notebook — wind/odor-regime min-EV histograms, and coloring PCA neural
trajectories by min-EV — apply verbatim from its "Plot minEV hist by wind odor regimes"
section onward. The only change needed there is the state column names (`phi_dot` is now
available as an estimated state).

## Summary of compatibility caveats

* Trained-agent logs must come from `action_physics='force'` (3-dim actions) with the **same
  physics yaml** as `force_physics` above; the integrator-replay check cell enforces this.
* `action_delay_const` / `action_latency` must have been off at eval, otherwise the logged raw
  actions are not the applied actions.
* The pybounds simulation differs from the env by the env's O(`sub_dt`) semi-implicit-Euler
  error (~0.5% of signal scale for fly.yaml time constants) — quantified in the synthetic
  verification cell.
* Within an env step, wind is zero-order-held; the `u_zeta_dot`/`u_w_dot` inputs reproduce the
  logged per-step wind sequence but smear each wind jump over one `dt`.
* The apparent-wind measurement is a genuine function of the state here (unlike the kinematic
  dynamic-wind system where `appWind = -u_para` was input feedback), so wind observability
  conclusions can differ qualitatively between the two systems even on identical trajectories.